In [1]:
import numpy as np
from matplotlib import pyplot as plt
import pandas as pd
from sklearn.metrics import confusion_matrix
import pynapple as nap
from spatial_manifolds.toroidal import *
from spatial_manifolds.behaviour_plots import *

from spatial_manifolds.mlencoding import *
from spatial_manifolds.circular_decoder import circular_decoder, cross_validate_decoder, cross_validate_decoder_time, circular_nanmean
from spatial_manifolds.data.curation import curate_clusters
from scipy.stats import zscore
from spatial_manifolds.util import gaussian_filter_nan
from spatial_manifolds.predictive_grid import compute_travel_projected, wrap_list
from spatial_manifolds.behaviour_plots import *
from spatial_manifolds.detect_grids import *
from spatial_manifolds.brainrender_helper import *

import warnings
warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2
%matplotlib inline

def read_yaml_file(file_path):
    import yaml
    with open(file_path, 'r') as file:
        data = yaml.safe_load(file)
    return data


In [2]:
GC_dir_path = '/Users/harryclark/Documents/data/xgboost_GC_assay/xgboost_assay_GC'
NGS_dir_path = '/Users/harryclark/Documents/data/xgboost_NGS_assay/xgboost_assay_NGS'

In [3]:
fig_path = '/Users/harryclark/Documents/figs/FIGURE2/'
# good examples include 
#mice = [25, 25, 26, 27, 29, 28]
#days = [25, 24, 18, 26, 23, 25]

In [25]:
n_neurons = np.arange(1, 500, 2)
n_neurons = np.insert(n_neurons, 0, 0) # we want the condition where no grid cells are used as a covariate history as well


In [ ]:
data = pd.DataFrame()
for dir_path, trained_on in zip([GC_dir_path, NGS_dir_path], ['GC', 'NGS']):
    for file in os.listdir(dir_path):
        if file.endswith('.yaml'):
            df = pd.DataFrame()
            mouse = file.split('_')[4]
            day = file.split('_')[5].split('day')[1]
            yaml_file_path = os.path.join(dir_path, file)
            yaml_data = read_yaml_file(yaml_file_path)
            tested_on = file.split('_')[-1].split('.yaml')[0]
            
            print(f'Processing mouse {mouse}, day {day}, trained on {trained_on}, tested on {tested_on}')
            for i, (key, values) in enumerate(yaml_data.items()):
                for j, val in enumerate(values):
                    df = pd.DataFrame()
                    df['mouse'] = [mouse]
                    df['day'] = [day]
                    df['trained_on'] = [trained_on]
                    df['tested_on'] = [tested_on]
                    df['n_neurons'] = [n_neurons[j]]
                    df['pR2'] = [val]
                    df['cluster_id'] = [key]
                    data = pd.concat([data, df], ignore_index=True)


Processing mouse 29, day 17, trained on GC, tested on cmGC
Processing mouse 21, day 26, trained on GC, tested on NGS
Processing mouse 21, day 18, trained on GC, tested on ncmGC
Processing mouse 29, day 19, trained on GC, tested on NGS
Processing mouse 21, day 24, trained on GC, tested on cmGC
Processing mouse 22, day 41, trained on GC, tested on NGS
Processing mouse 20, day 16, trained on GC, tested on cmGC
Processing mouse 29, day 22, trained on GC, tested on cmGC
Processing mouse 21, day 19, trained on GC, tested on NGS
Processing mouse 21, day 18, trained on GC, tested on NGS
Processing mouse 22, day 36, trained on GC, tested on NGS
Processing mouse 20, day 23, trained on GC, tested on cmGC
Processing mouse 26, day 12, trained on GC, tested on NGS
Processing mouse 22, day 34, trained on GC, tested on cmGC
Processing mouse 27, day 23, trained on GC, tested on ncmGC
Processing mouse 21, day 22, trained on GC, tested on ncmGC
Processing mouse 21, day 15, trained on GC, tested on ncmGC


In [38]:
df = pd.DataFrame()
df['mouse'] = mouse
df['day'] = day
df['trained_on'] = trained_on
df['tested_on'] = tested_on
df['n_neurons'] = n_neurons[j]
df['pR2'] = val
df['cluster_id'] = key

In [39]:
df

,mouse,day,trained_on,tested_on,n_neurons,pR2,cluster_id


In [6]:
yaml_data


{
    '102': [0.013810527629255697, 0.012321269440291804, 0.012739759576694076],
    '14': [0.008287772324249699, 0.006414680095212577, 0.008479286796364482],
    '19': [0.0066487652073455306, 0.00608786841930462, 0.00799441353688043],
    '24': [0.00807172570138368, 0.0075337767756763485, 0.01317395282710091],
    '73': [0.0585255687338536, 0.05660294845328466, 0.05997895977945149]
}

In [ ]:
n_neurons = np.arange(1, len(cov_cell_population_cluster_ids), 2)
n_neurons = np.insert(n_neurons, 0, 0) # we want the condition where no grid cells are used as a covariate history as well


In [ ]:
# loop over files in a /Users/harryclark/Documents/data/xgboost_GC_assay and read the yaml files


# read the yaml file
for file in os.listdir('/Users/harryclark/Documents/data/xgboost_GC_assay'):
    if file.endswith('.yaml'):
        yaml_file_path = os.path.join('/Users/harryclark/Documents/data/xgboost_GC_assay', file)
        yaml_data = read_yaml_file(yaml_file_path)
        print(f"Data from {file}:")
        print(yaml_data)
yaml_file_path = '/Users/harryclark/Documents/data/xgboost_GC_assay/GC_assay.yaml'
